In [57]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from IPython.display import display
import openai
import getpass
import base64
import io
from groq import Groq

In [58]:
#installation in jupyter notebook cell 
%pip install openai
%pip install groq

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [59]:
#Define PDF conversion
pipeline_options = PdfPipelineOptions()
pipeline_options.images_scale = 2
pipeline_options.generate_page_images =True

In [60]:
#Intialize the converter with these options
document_conveter = DocumentConverter(
    format_options = {InputFormat.PDF : PdfFormatOption(pipeline_options=pipeline_options)}
)

In [61]:
#Run the conversation
source = "/home/houssem_mabrouk/projects/prompt_chaining_task_routing/pdf_invoice.pdf"
result = document_conveter.convert(source)


Loading weights: 100%|██████████| 770/770 [00:00<00:00, 1818.84it/s]


In [62]:
#Extract text from he document
doc = result.document
text = doc.export_to_markdown()

In [63]:
#Set up Groq API
api_key = getpass.getpass("Enter your Groq API key: ")
client = Groq(api_key=api_key)


In [64]:
#creation of function to ask questions to the model
def ask_question_openai(model,system_content="You are a helpful assistant",user_content=None,text=None):
                 response =client.responses.create(
                     model =model,
                     instructions = system_content,
                     input = user_content
                 )
                 return response.output_text 

In [65]:
def ask_question(model, system_content="You are a helpful assistant",
                 user_content=None, text=None):

    response = client.chat.completions.create(
        model=model,
       messages=[
           {"role": "system", "content": system_content},
           {"role": "user", "content": user_content}
        ],
       
        
    )

    return response.choices[0].message.content

In [66]:
#asking about the invoice number in the pdf document
question_1 = f"What is the invoice number of:{text}"
answer_1 = ask_question(model = "openai/gpt-oss-20b", user_content = question_1)
print(f"Answer:{answer_1}")

Answer:The invoice number is **16384**.


In [67]:
#asking about the balance due in the pdf document
question_2 = f"what is the total balance due of :{text}"
answer_2 = ask_question(model = "openai/gpt-oss-20b",user_content = question_2)
print(f"Answer:{answer_2}")

Answer:The total balance due on the invoice is **$6,208.84**.


In [68]:
#creation of a function to encode image of page in base64
def encode_image(doc,page):
    buffered = io.BytesIO()
    doc.pages[page].image.pil_image.save(buffered,format="PNG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")



In [69]:
#encode the image as base64 string and print it
base64_string = encode_image(doc,1)
print("Base64 string representation of page 1 image :",base64_string)

Base64 string representation of page 1 image : ...


In [70]:
#extract the invoice date from the image

def ask_question(model, system_content="You are a helpful assistant",
                 user_content=None, text=None):

    if isinstance(user_content, list):
        messages = [
            {"role": "system", "content": system_content}
        ] + user_content
    else:
        messages = [
            {"role": "system", "content": system_content},
            {"role": "user", "content": user_content}
        ]

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )

    return response.choices[0].message.content



In [71]:
question_3 = "Extract the invoice date from the image."

input_3 = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": question_3
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{base64_string}"
                }
            }
        ]
    }
]


In [72]:
question_3  = ask_question(model="qwen/qwen3.8-27b",user_content = input_3)
print(f"Answer:{question_3}")

Answer:Dec 08 2012


In [73]:
question_4 = "What is the shipping address on the invoice?"

input_4 = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": question_4
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{base64_string}"
                }
            }
        ]
    }
]

In [74]:
question_4  = ask_question(model="qwen/qwen3.8-27b",user_content = input_4)
print(f"Answer:{question_4}")

Answer:Based on the invoice provided, the shipping address is listed under the "Ship To:" section.

The shipping address is:
**Nottingham, England, United Kingdom**

---

**Note:** The invoice does not provide a full street address (e.g., house number, street name, or postal code). It only lists the city, country, and country name. This is common for some types of invoices or when the customer's full address is not required for billing purposes. For actual shipping, a more detailed address would typically be needed.
